In [17]:
import json
import warnings
import pandas as pd

pd.set_option('future.no_silent_downcasting', True)
pd.set_option('display.max_columns', None)

warnings.filterwarnings('ignore')

In [18]:
df = pd.read_csv('powerBI/raw_data_with_predictions.csv')

In [19]:
vehicle_price_mapping = {
    'less than 20000': 20000,
    '20000 to 29000': 25000,
    '30000 to 39000': 35000,
    '40000 to 59000': 50000,
    '60000 to 69000': 65000,
    'more than 69000': 75000
}

df['VehiclePrice_average'] = df['VehiclePrice'].map(vehicle_price_mapping)

In [20]:
# df.FraudFound_P.value_counts()

In [21]:
df.to_csv("powerBI/raw_data.csv", index=False)

---

## EBM local scores

In [36]:
with open('powerBI/all_perils_ebm.json', 'r') as f:
    all_perils_ebm = json.load(f)

with open('powerBI/collision_ebm.json', 'r') as f:
    collision_ebm = json.load(f)

In [37]:
all_perils_ebm['ebm'].keys()

dict_keys(['outputs', 'intercept', 'bagged_intercept', 'bag_weights', 'best_iteration', 'implementation', 'implementation_params', 'features', 'terms'])

In [43]:
ap_category_db = pd.DataFrame()
ap_numeric_db = pd.DataFrame()

all_p_intercept = all_perils_ebm['ebm']['intercept']

for feature, term in zip(all_perils_ebm['ebm']['features'], all_perils_ebm['ebm']['terms']):
    if 'categories' in list(feature.keys()):
        print(feature['name'])
        value_df = pd.DataFrame(feature['categories']).T
        value_df.columns=['input_value']
        value_df['feature'] = feature['name']

        value_df['score'] = term['scores'][1:-1]
        value_df['score_std'] = term['standard_deviations'][1:-1]

        ap_category_db = pd.concat([ap_category_db, value_df], axis=0)
        
    else:
        print(">", feature['name'])
        # numerical feature들을 'binning'하여 score를 부여한 형태
        total_feature_cuts = [feature['min']] + feature['cuts'][0] + [feature['max']]
        tmp = pd.DataFrame(total_feature_cuts, columns=['cut_start'])
        tmp['cut_end'] = tmp.cut_start.shift(-1)
        tmp.dropna(inplace=True)
        tmp['feature'] = feature['name']

        tmp['score'] = term['scores'][1:-1]
        tmp['score_std'] = term['standard_deviations'][1:-1]
        # 따라서 각각의 bin에 따라 score가 분배 되어 있다
        tmp_df_list = list()

        for value in df[feature['name']].unique():
            extracted_tmp = tmp.loc[(tmp['cut_start']<=value) & (tmp['cut_end']>value)]
            extracted_tmp['input_value'] = value
            tmp_df_list.append(extracted_tmp)
        # 가장 마지막 value 값도 추가
        extracted_tmp = tmp.loc[tmp['cut_end']==feature['max']]
        extracted_tmp['input_value'] = feature['max']
        tmp_df_list.append(extracted_tmp)

        value_df = pd.concat(tmp_df_list, axis=0)
        ap_numeric_db = pd.concat([ap_numeric_db, value_df], axis=0)

        # if feature['name']=="DriverRating":
        #     break
        
ap_numeric_db.sort_values(['feature', 'input_value'], inplace=True)
ap_numeric_db.drop(columns=['cut_start', 'cut_end'], inplace=True)
ap_category_db['intercept'] = all_p_intercept[0]
ap_numeric_db['intercept'] = all_p_intercept[0]

Make
VehiclePrice_num
VehicleCategory
AgeOfVehicle
Sex
MaritalStatus
> Age
> DriverRating
AgeOfPolicyHolder
NumberOfCars
PastNumberOfClaims
PolicyType
> Deductible
AgentType
NumberOfSuppliments
Days_Policy_Accident
Days_Policy_Claim
PoliceReportFiled
WitnessPresent
AddressChange_Claim_2


({'name': 'Make',
  'type': 'nominal',
  'num_unique_vals': 17,
  'categories': [['Accura',
    'BMW',
    'Chevrolet',
    'Dodge',
    'Ford',
    'Honda',
    'Jaguar',
    'Mazda',
    'Mecedes',
    'Mercury',
    'Nisson',
    'Pontiac',
    'Porche',
    'Saab',
    'Saturn',
    'Toyota',
    'VW']]},
 {'term_features': ['Make'],
  'scores': [0.0,
   0.4723664744490646,
   -1.0410160842816447,
   -0.1874101227123349,
   -0.4046564910649314,
   -0.25190676874481177,
   -0.5325484535042988,
   -1.0383527608940952,
   -0.08763388926604124,
   -1.0045415449907833,
   -0.6121179225027065,
   -0.9721358247219473,
   0.3321393501865147,
   -0.9609770790952449,
   0.13529266805406057,
   -0.5195154880208418,
   -0.0014078569924194256,
   -0.4321577308259623,
   0.0],
  'standard_deviations': [0.0,
   0.061206813442847216,
   0.3870603545946905,
   0.04968338977445204,
   0.29716331220748643,
   0.06930045894769435,
   0.04273308196195984,
   0.39116254363251063,
   0.04261302345285889,

In [25]:
ap_numeric_db['policy_type'] = "All Perils"
ap_numeric_db['feature_type'] = "numeric"
ap_category_db['policy_type'] = "All Perils"
ap_category_db['feature_type'] = "category"

all_p_local_scores = pd.concat([ap_numeric_db, ap_category_db], axis=0)

In [45]:
all_p_local_scores

,feature,score,score_std,input_value,intercept,policy_type,feature_type
0,Age,1.141860,0.506628,0.0,-1.211982,All Perils,numeric
1,Age,1.129908,0.506179,16.0,-1.211982,All Perils,numeric
1,Age,1.129908,0.506179,17.0,-1.211982,All Perils,numeric
1,Age,1.129908,0.506179,18.0,-1.211982,All Perils,numeric
2,Age,1.623872,1.111519,19.0,-1.211982,All Perils,numeric
...,...,...,...,...,...,...,...
1,PoliceReportFiled,-2.259771,0.228416,Yes,-1.211982,All Perils,category
0,WitnessPresent,0.008652,0.003212,No,-1.211982,All Perils,category
1,WitnessPresent,-1.400391,0.519933,Yes,-1.211982,All Perils,category
0,AddressChange_Claim_2,-0.074757,0.281110,change,-1.211982,All Perils,category


In [27]:
c_category_db = pd.DataFrame()
c_numeric_db = pd.DataFrame()

collision_intercept = collision_ebm['ebm']['intercept']

for feature, term in zip(collision_ebm['ebm']['features'], collision_ebm['ebm']['terms']):
    if 'categories' in list(feature.keys()):
        print(feature['name'])
        value_df = pd.DataFrame(feature['categories']).T
        value_df.columns=['input_value']
        value_df['feature'] = feature['name']

        value_df['score'] = term['scores'][1:-1]
        value_df['score_std'] = term['standard_deviations'][1:-1]

        c_category_db = pd.concat([c_category_db, value_df], axis=0)
    else:
        print(">", feature['name'])
        # numerical feature들을 'binning'하여 score를 부여한 형태
        total_feature_cuts = [feature['min']] + feature['cuts'][0] + [feature['max']]
        tmp = pd.DataFrame(total_feature_cuts, columns=['cut_start'])
        tmp['cut_end'] = tmp.cut_start.shift(-1)
        tmp.dropna(inplace=True)
        tmp['feature'] = feature['name']

        tmp['score'] = term['scores'][1:-1]
        tmp['score_std'] = term['standard_deviations'][1:-1]
        # 따라서 각각의 bin에 따라 scorer가 분배 되어 있다
        tmp_df_list = list()

        for value in df[feature['name']].unique():
            extracted_tmp = tmp.loc[(tmp['cut_start']<=value) & (tmp['cut_end']>value)]
            extracted_tmp['input_value'] = value
            tmp_df_list.append(extracted_tmp)
        # 가장 마지막 value 값도 추가
        extracted_tmp = tmp.loc[tmp['cut_end']==feature['max']]
        extracted_tmp['input_value'] = feature['max']
        tmp_df_list.append(extracted_tmp)

        value_df = pd.concat(tmp_df_list, axis=0)
        c_numeric_db = pd.concat([c_numeric_db, value_df], axis=0)
        
c_numeric_db.sort_values(['feature', 'input_value'], inplace=True)
c_numeric_db.drop(columns=['cut_start', 'cut_end'], inplace=True)
c_category_db['intercept'] = collision_intercept[0]
c_numeric_db['intercept'] = collision_intercept[0]

Make
VehiclePrice_num
VehicleCategory
AgeOfVehicle
Sex
MaritalStatus
> Age
> DriverRating
AgeOfPolicyHolder
NumberOfCars
PastNumberOfClaims
PolicyType
> Deductible
AgentType
NumberOfSuppliments
Days_Policy_Accident
Days_Policy_Claim
PoliceReportFiled
WitnessPresent
AddressChange_Claim_2


In [28]:
c_numeric_db['policy_type'] = "Collision"
c_numeric_db['feature_type'] = "numeric"
c_category_db['policy_type'] = "Collision"
c_category_db['feature_type'] = "category"

collision_local_scores = pd.concat([c_numeric_db, c_category_db], axis=0)

In [29]:
collision_local_scores

,feature,score,score_std,input_value,intercept,policy_type,feature_type
0,Age,0.680777,0.421597,0.0,-1.544721,Collision,numeric
1,Age,0.688154,0.419041,16.0,-1.544721,Collision,numeric
2,Age,0.784149,0.562550,17.0,-1.544721,Collision,numeric
3,Age,0.739739,0.469851,18.0,-1.544721,Collision,numeric
4,Age,0.725152,0.457805,19.0,-1.544721,Collision,numeric
...,...,...,...,...,...,...,...
1,PoliceReportFiled,-1.656425,0.208260,Yes,-1.544721,Collision,category
0,WitnessPresent,0.009074,0.001389,No,-1.544721,Collision,category
1,WitnessPresent,-1.369445,0.209649,Yes,-1.544721,Collision,category
0,AddressChange_Claim_2,0.181922,0.143245,change,-1.544721,Collision,category


In [46]:
local_scores = pd.concat([collision_local_scores, all_p_local_scores], axis=0)
local_scores.to_csv("powerBI/local_scores.csv", index=False)

---

In [47]:
local_scores

,feature,score,score_std,input_value,intercept,policy_type,feature_type
0,Age,0.680777,0.421597,0.0,-1.544721,Collision,numeric
1,Age,0.688154,0.419041,16.0,-1.544721,Collision,numeric
2,Age,0.784149,0.562550,17.0,-1.544721,Collision,numeric
3,Age,0.739739,0.469851,18.0,-1.544721,Collision,numeric
4,Age,0.725152,0.457805,19.0,-1.544721,Collision,numeric
...,...,...,...,...,...,...,...
1,PoliceReportFiled,-2.259771,0.228416,Yes,-1.211982,All Perils,category
0,WitnessPresent,0.008652,0.003212,No,-1.211982,All Perils,category
1,WitnessPresent,-1.400391,0.519933,Yes,-1.211982,All Perils,category
0,AddressChange_Claim_2,-0.074757,0.281110,change,-1.211982,All Perils,category


In [48]:
products = df[['BasePolicy']].drop_duplicates()
products.to_csv("powerBI/products.csv", index=True)
products

,BasePolicy
0,Liability
1,Collision
9,All Perils


---

In [52]:
time_vars = ["Month", "WeekOfMonth", "DayOfWeek", "DayOfWeekClaimed", 'MonthClaimed', 'WeekOfMonthClaimed', 'Year']
vehicle_vars = ["Make", "VehiclePrice_num", "VehicleCategory", "AgeOfVehicle", "VehiclePrice", "VehiclePrice_average"]
personal_vars = ["Sex", 'MaritalStatus', "Age", 'DriverRating', 'AgeOfPolicyHolder', 'NumberOfCars', 'PastNumberOfClaims', 'AddressChange_Claim']
policy_vars = ["PolicyType", 'Deductible', 'AgentType', "NumberOfSuppliments", 'PolicyNumber', 'RepNumber', 'BasePolicy']
accident_vars = ['Days_Policy_Accident', 'Days_Policy_Claim', 'PoliceReportFiled', 'WitnessPresent', 'AddressChange_Claim_2', 'AccidentArea', 'Fault', 'FraudFound_P', 'predictions']

In [53]:
time_df = pd.DataFrame(time_vars, columns=['feature'])
time_df['feature_type'] = 'time'

vehicle_df = pd.DataFrame(vehicle_vars, columns=['feature'])
vehicle_df['feature_type'] = 'vehicle'

personal_df = pd.DataFrame(personal_vars, columns=['feature'])
personal_df['feature_type'] = 'personal'

policy_df = pd.DataFrame(policy_vars, columns=['feature'])
policy_df['feature_type'] = 'policy'

accident_df = pd.DataFrame(accident_vars, columns=['feature'])
accident_df['feature_type'] = 'accident'

features = pd.concat([time_df, vehicle_df, personal_df, policy_df, accident_df], axis=0)

In [54]:
features

,feature,feature_type
0,Month,time
1,WeekOfMonth,time
2,DayOfWeek,time
3,DayOfWeekClaimed,time
4,MonthClaimed,time
5,WeekOfMonthClaimed,time
6,Year,time
0,Make,vehicle
1,VehiclePrice_num,vehicle
2,VehicleCategory,vehicle


In [55]:
features.to_csv("powerBI/features.csv", index=False)
features.tail(3)

,feature,feature_type
6,Fault,accident
7,FraudFound_P,accident
8,predictions,accident


---